### Produce sheets and pdf plots for RLRT PRD paper

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
import sys
sys.path.append('/Users/zihaolin/Desktop/Neutrino/RLRTworkdir/RLRTExtraction')

from analysis.plotting import plot_response_qvbin, plot_response_q2bin
from analysis.extraction import prepare_dataframe, calculate_bin_centering_correction, extract_response_qvbins, extract_response_q2bins
from analysis.presets import QVCENTERS, Q2CENTERS, ALPHA_FINE
from analysis.christy_bodek_fit import calculate_response_table, special_sigmoid
from analysis.plotting import *



### Prepare dataframe for $R_L$ $R_T$ extraction

In [2]:
# # compute from scratch
# df = pd.read_csv('/Users/zihaolin/Desktop/Neutrino/RLRTworkdir/RLRTExtraction/Carbon/12C_data_raw/12C-25Sep14.csv')
# df = df.loc[df['nu']>0]
# df = prepare_dataframe(df_data = df)
# df = calculate_bin_centering_correction(df_xsec = df)
# df.to_csv('/Users/zihaolin/Desktop/Neutrino/RLRTworkdir/RLRTExtraction/Carbon/12C_data_raw/12C-26Sep25-bin-centering.csv', index=False)


### Produce $R_L$ $R_T$ extraction, this analysis

In [3]:
# # load from pre computed data frame
# df = pd.read_csv('/Users/zihaolin/Desktop/Neutrino/RLRTworkdir/RLRTExtraction/Carbon/12C_data_raw/12C-26Sep25-bin-centering.csv')

# # drop Dai and e4nu
# df = df.loc[df['dataSet']!=6] # drop Dai
# df = df.loc[df['dataSet']!=25] # drop e4nu

# # extract in qv bin 
# df_rlrt = extract_response_qvbins(df, qvcenters = QVCENTERS)
# with pd.ExcelWriter('/Users/zihaolin/Desktop/Neutrino/RLRTworkdir/RLRTExtraction/Carbon/one_sheet_to_rule_them_all.xlsx',
#                     engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
#     df_rlrt.to_excel(writer, sheet_name='this_analysis_qvbin', index=False)

# # extract in q2 bin 
# df_rlrt = extract_response_q2bins(df, q2centers = Q2CENTERS)
# with pd.ExcelWriter('/Users/zihaolin/Desktop/Neutrino/RLRTworkdir/RLRTExtraction/Carbon/one_sheet_to_rule_them_all.xlsx',
#                     engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
#     df_rlrt.to_excel(writer, sheet_name='this_analysis_q2bin', index=False)


### Produce Christy-Bodek fit

In [4]:
# # produce qv bins
# df_CB = []
# for qv in QVCENTERS:
#     df_qv = pd.DataFrame({'nu':np.linspace(0.001, qv*0.999, 500)})

#     df_qv['qv'] = qv
#     df_qv = calculate_response_table(df_qv)
#     for col in ["rttot", "rltot", "rtqe", "rlqe","rtie", "rlie", "rte", "rle", "rtns", "rlns", "rtqd", "rlqd"]:
#         df_qv[col]*=1e3
#     df_CB.append(df_qv)
#     print(f'qv = {qv} done')
# df_CB = pd.concat(df_CB)
# with pd.ExcelWriter('/Users/zihaolin/Desktop/Neutrino/RLRTworkdir/RLRTExtraction/Carbon/one_sheet_to_rule_them_all.xlsx',
#                     engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
#     df_CB.to_excel(writer, sheet_name='CBfit_qvbin', index=False)


# # produce q2 bins
# df_CB = []
# nu_max = [0.5, 0.5,0.5, 0.6, 0.6,
#     0.7, 0.7, 0.7, 2.2, 2.2,
#     2.5, 3.0, 3.0, 3.5, 3.5,
#     3.8, 3.8, 3.8]
# for i in range(len(Q2CENTERS)):
#     q2 = Q2CENTERS[i]
#     df_q2 = pd.DataFrame({'nu':np.linspace(0.001, nu_max[i], 500)})
#     df_q2['qv'] = np.sqrt(q2 + df_q2['nu']**2)
#     df_q2 = calculate_response_table(df_q2)
#     for col in ["rttot", "rltot", "rtqe", "rlqe","rtie", "rlie", "rte", "rle", "rtns", "rlns", "rtqd", "rlqd"]:
#         df_q2[col]*=1e3
#     df_q2['q2'] = q2
#     df_CB.append(df_q2)
#     print(f'q2 = {q2} done')
# df_CB = pd.concat(df_CB)
# with pd.ExcelWriter('/Users/zihaolin/Desktop/Neutrino/RLRTworkdir/RLRTExtraction/Carbon/one_sheet_to_rule_them_all.xlsx',
#                     engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
#     df_CB.to_excel(writer, sheet_name='CBfit_q2bin', index=False)


### Produce Christy-Bodek fit photo-absorption cross section

In [5]:
# # produce CBfit RT at q2=0.001

# q2 = 0.001
# inputs = pd.DataFrame({'nu':np.arange(0.001, 2.61, 0.001)})

# inputs['qv'] = np.sqrt(q2 + inputs['nu']**2)
# df_CB = calculate_response_table(inputs)
# df_CB['q2'] = q2

# df_CB['nu (MeV)'] = df_CB['nu'] * 1e3
# df_CB['RT (MeV^-1)'] = df_CB['rttot']
# df_CB['photon xsec (GeV^-2)'] = df_CB['rttot']*1e3 * 2 * (np.pi**2) * ALPHA_FINE / df_CB['nu'] # GeV^-2
# df_CB['photon xsec (mb)'] = df_CB['photon xsec (GeV^-2)'] * 0.389379 # convert to mb

# sheet_exp_rt = pd.read_excel('/Users/zihaolin/Desktop/Neutrino/RLRTworkdir/RLRTExtraction/Carbon/one_sheet_to_rule_them_all.xlsx', sheet_name='exp_rt_q2bin')
# sheet_exp_rt = sheet_exp_rt.loc[sheet_exp_rt['experiment']=='Photo-production']
# sheet_exp_rt['nu (MeV)'] = sheet_exp_rt['nu'] * 1e3
# sheet_exp_rt['photon RT (MeV^-1)'] = sheet_exp_rt['rt'] * 1e-3
# sheet_exp_rt['photon RT error (MeV^-1)'] = sheet_exp_rt['rterr'] * 1e-3
# sheet_exp_rt['photon xsec data (GeV^-2)'] = sheet_exp_rt['rt'] * 2 * (np.pi**2) * ALPHA_FINE / sheet_exp_rt['nu'] # GeV^-2
# sheet_exp_rt['photon xsec data error (GeV^-2)'] = sheet_exp_rt['rterr'] * 2 * (np.pi**2) * ALPHA_FINE / sheet_exp_rt['nu'] # GeV^-2
# sheet_exp_rt['photon xsec data (mb)'] = sheet_exp_rt['photon xsec data (GeV^-2)'] * 0.389379 # convert to mb
# sheet_exp_rt['photon xsec data error (mb)'] = sheet_exp_rt['photon xsec data error (GeV^-2)'] * 0.389379 # convert to mb

# # write to one_sheet_to_rule_them_all.xlsx
# with pd.ExcelWriter('/Users/zihaolin/Desktop/Neutrino/RLRTworkdir/RLRTExtraction/Carbon/one_sheet_to_rule_them_all.xlsx',
#     mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
#     df_CB.to_excel(writer, sheet_name='CBfit_photo_xsec', index=False)
#     sheet_exp_rt.to_excel(writer, sheet_name='exp_photo_xsec', index=False)


# # write to CBfit_q2_0.001.xlsx
# with pd.ExcelWriter('/Users/zihaolin/Desktop/Neutrino/RLRTworkdir/RLRTExtraction/Carbon/12C_plots/CBfit_q2_0.001.xlsx',
#     mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
#     df_CB.loc[df_CB['nu']<=0.62].to_excel(writer, sheet_name='CBfit_photo_xsec', index=False) # plotting stops at 620 MeV
#     sheet_exp_rt.to_excel(writer, sheet_name='exp_photo_xsec', index=False)



In [6]:
# # plot photo-absorbtion RT, xsec
# q2 = 0.001
# df_CB = pd.read_excel('/Users/zihaolin/Desktop/Neutrino/RLRTworkdir/RLRTExtraction/Carbon/one_sheet_to_rule_them_all.xlsx', sheet_name='CBfit_photo_xsec')
# df_CB = df_CB[df_CB['nu']<=0.620]
# sheet_exp_rt = pd.read_excel('/Users/zihaolin/Desktop/Neutrino/RLRTworkdir/RLRTExtraction/Carbon/one_sheet_to_rule_them_all.xlsx', sheet_name='exp_photo_xsec')

# sheet_exp_rt = sheet_exp_rt.loc[sheet_exp_rt['nu (MeV)'] <= 620]
# plt.figure(dpi=200, figsize=(10,4))
# plt.errorbar(sheet_exp_rt['nu (MeV)'],sheet_exp_rt['photon RT (MeV^-1)'],yerr=sheet_exp_rt['photon RT error (MeV^-1)'], label='data', fmt='o', ms=3, color='limegreen', alpha=0.5)
# plt.plot(df_CB['nu (MeV)'], df_CB['RT (MeV^-1)'], label=f'CBfit RT $Q^2$={q2}GeV$^2$',linestyle=':',color='black', alpha=0.5)
# plt.xlim(0, 620)
# plt.ylim(0, None)
# plt.xlabel('$\\nu$ (MeV)')
# plt.ylabel('$R_T$ (MeV$^{-1}$)')
# plt.legend()
# plt.show()


# plt.figure(dpi=200, figsize=(10,4))
# plt.errorbar(sheet_exp_rt['nu (MeV)'],sheet_exp_rt['photon xsec data (mb)'],yerr=sheet_exp_rt['photon xsec data error (mb)'], label='data', fmt='o', ms=3, color='limegreen', alpha=0.5)
# plt.plot(df_CB['nu (MeV)'], df_CB['photon xsec (mb)'], label=f'CBfit photo xsec $Q^2$={q2}GeV$^2$',linestyle=':',color='black', alpha=0.5)
# plt.xlim(0, 620)
# plt.ylim(0, None)
# plt.xlabel('$\\nu$ (MeV)')
# plt.ylabel('photon xsec (mb)')
# plt.legend()
# plt.show()



### Produce Q2 bin ED-RMF (QE) + SuSAv2 (2p2h)

In [7]:
# # qvbin: produce sum using interpolation
# plot_xlsx = '/Users/zihaolin/Desktop/Neutrino/RLRTworkdir/RLRTExtraction/Carbon/one_sheet_to_rule_them_all.xlsx'
# rlrt = 'rt'
# sheet_theory = pd.read_excel(plot_xlsx, sheet_name=f'theory_{rlrt}_qvbin')
# dfs = []
# for qvcenter in QVCENTERS[0:12]:
#     edrmf = sheet_theory.loc[(sheet_theory['theory']=='ED-RMF') & (sheet_theory['qv']==qvcenter)]
#     x1s = edrmf['nu']
#     y1s = edrmf[f'{rlrt}']

#     susa = sheet_theory.loc[(sheet_theory['theory']=='SuSAv2') & (sheet_theory['qv']==qvcenter)]
#     x2s = susa['nu']
#     y2s = susa[f'{rlrt}2p2h']

#     f_2p2h = interp1d(
#         x2s,
#         y2s,
#         kind="linear",
#         bounds_error=False,
#         fill_value=0.0
#     )

#     y2s_on_x1 = f_2p2h(x1s)

#     # QE + 2p2h
#     x_total = x1s
#     y_total = y1s + y2s_on_x1
#     plt.figure()

#     plt.plot(x1s,np.zeros_like(x1s), linestyle='--', color='gray', alpha=0.5)

#     plt.plot(x1s,y1s, label='EDRMF QE')
#     plt.plot(susa['nu'],susa[f'{rlrt}qe'], label='SuSAv2 QE')
#     plt.plot(x2s,y2s, label='SuSAv2 2p2h')
#     plt.plot(susa['nu'],susa[f'{rlrt}inel'], label='SuSAv2 inelastic')
#     plt.plot(susa['nu'],susa[f'{rlrt}qe']+susa[f'{rlrt}2p2h'], label='SuSAv2 QE+2p2h')
#     plt.plot(x_total,y_total, label='EDRMF(QE)+SuSAv2(2p2h)')
#     plt.legend()
#     plt.xlabel('$\\nu$')
#     plt.ylabel('$R_T$')
#     plt.title(f'q={qvcenter}')
#     plt.show()
#     plt.close()
#     df = pd.DataFrame()

#     df['nu'] = x_total
#     df[f'{rlrt}'] = y_total
#     df['qv'] = qvcenter
#     df['theory'] = 'EDRMF(QE)+SuSAv2(2p2h)'
#     dfs.append(df)

# dfs = pd.concat(dfs)

# # # write to one_sheet_to_rule_them_all.xlsx
# # with pd.ExcelWriter(plot_xlsx, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
# #     dfs_original = pd.read_excel(plot_xlsx, sheet_name=f'theory_{rlrt}_qvbin')
# #     dfs_original = dfs_original.loc[dfs_original['theory']!='EDRMF(QE)+SuSAv2(2p2h)']

# #     dfs_original = pd.concat([dfs_original, dfs])
# #     dfs_original.to_excel(writer, sheet_name=f'theory_{rlrt}_qvbin', index=False)



In [8]:
# # q2bin: produce sum using interpolation
# plot_xlsx = '/Users/zihaolin/Desktop/Neutrino/RLRTworkdir/RLRTExtraction/Carbon/one_sheet_to_rule_them_all.xlsx'
# rlrt = 'rt'
# sheet_theory = pd.read_excel(plot_xlsx, sheet_name=f'theory_{rlrt}_q2bin')
# dfs = []
# for q2center in Q2CENTERS[0:13]:
#     edrmf = sheet_theory.loc[(sheet_theory['theory']=='ED-RMF') & (sheet_theory['q2']==q2center)]
#     x1s = edrmf['nu']
#     y1s = edrmf[f'{rlrt}']
#     susa = sheet_theory.loc[(sheet_theory['theory']=='SuSAv2') & (sheet_theory['q2']==q2center)]
#     x2s = susa['nu']
#     y2s = susa[f'{rlrt}2p2h']

#     f_2p2h = interp1d(
#         x2s,
#         y2s,
#         kind="linear",
#         bounds_error=False,
#         fill_value=0.0
#     )

#     y2s_on_x1 = f_2p2h(x1s)

#     # QE + 2p2h
#     x_total = x1s
#     y_total = y1s + y2s_on_x1
#     plt.figure()

#     plt.plot(x1s,np.zeros_like(x1s), linestyle='--', color='gray', alpha=0.5)

#     plt.plot(x1s,y1s, label='EDRMF QE')
#     plt.plot(susa['nu'],susa[f'{rlrt}qe'], label='SuSAv2 QE')
#     plt.plot(x2s,y2s, label='SuSAv2 2p2h')
#     plt.plot(susa['nu'],susa[f'{rlrt}inel'], label='SuSAv2 inelastic')
#     plt.plot(susa['nu'],susa[f'{rlrt}qe']+susa[f'{rlrt}2p2h'], label='SuSAv2 QE+2p2h')
#     plt.plot(x_total,y_total, label='EDRMF+SuSAv2')
#     plt.legend()
#     plt.xlabel('$\\nu$')
#     plt.ylabel('$R_T$')
#     plt.title(f'q2={q2center}')
#     plt.show()
#     plt.close()
#     df = pd.DataFrame()

#     df['nu'] = x_total
#     df[f'{rlrt}'] = y_total
#     df['q2'] = q2center
#     df['theory'] = 'EDRMF(QE)+SuSAv2(2p2h)'
#     dfs.append(df)

# dfs = pd.concat(dfs)

# # write to one_sheet_to_rule_them_all.xlsx
# with pd.ExcelWriter(plot_xlsx, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
#     dfs_original = pd.read_excel(plot_xlsx, sheet_name=f'theory_{rlrt}_q2bin')
#     dfs_original = dfs_original.loc[dfs_original['theory']!='EDRMF(QE)+SuSAv2(2p2h)']

#     dfs_original = pd.concat([dfs_original, dfs])
#     dfs_original.to_excel(writer, sheet_name=f'theory_{rlrt}_q2bin', index=False)



### Produce Christy-Bodek fit at low Ex, high resolution 

In [9]:
# # produce qv bins
# df_CB = []
# for qv in QVCENTERS:
    
#     exs = np.linspace(0.001, 0.08, 400)
#     nus = - MASS_C12 + np.sqrt(MASS_C12**2 + qv**2 + 2*MASS_C12 * exs)
#     df_qv = pd.DataFrame({'nu':nus})
#     df_qv['qv'] = qv
#     df_qv = calculate_response_table(df_qv)
#     for col in ["rttot", "rltot", "rtqe", "rlqe","rtie", "rlie", "rte", "rle", "rtns", "rlns", "rtqd", "rlqd"]:
#         df_qv[col]*=1e3
#     df_CB.append(df_qv)
#     print(f'qv = {qv} done')
# df_CB = pd.concat(df_CB)
# with pd.ExcelWriter('/Users/zihaolin/Desktop/Neutrino/RLRTworkdir/RLRTExtraction/Carbon/one_sheet_to_rule_them_all.xlsx',
#                     engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
#     df_CB.to_excel(writer, sheet_name='CBfit_qvbin_low_ex', index=False)


# # produce q2 bins
# df_CB = []
# for q2 in Q2CENTERS:
#     exs = np.linspace(0.001, 0.08, 400)
#     A = 12
#     nuel = q2/2./(0.931494*A)
#     nus = exs + nuel
#     df_q2 = pd.DataFrame({'nu':nus})
#     df_q2['qv'] = np.sqrt(q2 + df_q2['nu']**2)
#     df_q2 = calculate_response_table(df_q2)
#     for col in ["rttot", "rltot", "rtqe", "rlqe","rtie", "rlie", "rte", "rle", "rtns", "rlns", "rtqd", "rlqd"]:
#         df_q2[col]*=1e3
#     df_q2['q2'] = q2
#     df_CB.append(df_q2)
#     print(f'q2 = {q2} done')
# df_CB = pd.concat(df_CB)
# with pd.ExcelWriter('/Users/zihaolin/Desktop/Neutrino/RLRTworkdir/RLRTExtraction/Carbon/one_sheet_to_rule_them_all.xlsx',
#                     engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
#     df_CB.to_excel(writer, sheet_name='CBfit_q2bin_low_ex', index=False)


### Produce Christy-Bodek fit cross section for supplementary material

In [10]:
# df = pd.read_csv('/Users/zihaolin/Desktop/Neutrino/RLRTworkdir/RLRTExtraction/Carbon/12C_data_raw/12C-25Sep14.csv')
# df = df.loc[df['nu']>0]

# df_CB = []
# for ds in df['dataSet'].unique():
#     df_ds = df.loc[df['dataSet']==ds]
#     for E0 in df_ds['E0'].unique():
#         df_E0 = df_ds.loc[df_ds['E0']==E0]
#         for theta in df_E0['ThetaDeg'].unique():
#             df_theta = df_E0.loc[df_E0['ThetaDeg']==theta]
#             nu_range = (df_theta['nu'].max()-df_theta['nu'].min())
#             inputs = pd.DataFrame({'nu':
#                 np.linspace(0.001 if df_theta['nu'].min()<0.2 
#                     else max(df_theta['nu'].min()-nu_range*0.05, 0.001),
#                 df_theta['nu'].max()+nu_range*0.05,
#                 300)})
#             inputs['e']=E0
#             inputs['theta']=theta
#             df_input = calculate_cross_section_table(inputs)
#             df_CB.append(df_input)
#             plt.figure()
#             plt.plot(df_input['nu'], df_input['xs_total'] * 12*1e3)
#             normCrossError = df_theta['cross'] * NORMALIZATIONS[ds] * np.sqrt((df_theta['error']/df_theta['cross'])**2+(NORMALIZATION_ERRORS[ds]/NORMALIZATIONS[ds])**2)
#             normCross = df_theta['cross'] * NORMALIZATIONS[ds]
#             plt.errorbar(df_theta['nu'], normCross, yerr=normCrossError,fmt='o')
#             plt.title(f'ds={ds}, e={E0}, $\\theta$={theta}')
#             plt.show()
#             plt.close()
# df_CB = pd.concat(df_CB)
# with pd.ExcelWriter('/Users/zihaolin/Desktop/Neutrino/RLRTworkdir/RLRTExtraction/Carbon/one_sheet_to_rule_them_all.xlsx',
#                     engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
#     df_CB.to_excel(writer, sheet_name='CBfit_xsec', index=False)
